In [1]:
#!/usr/bin/env python3

# Phase 2 — Evaluate Model H (Hindi)
#

In [2]:
# === UPDATE THESE PATHS ===
CHECKPOINT_PATH = "/kaggle/input/notebooks/srinivasu2005/train-hindi/checkpoints_hindi/best_model.pt"
TEST_BIN = "/kaggle/input/notebooks/srinivasu2005/tokenize-hindi/hindi_tokenized/test.bin"
TEST_TEXT = "/kaggle/input/datasets/srinivasu2005/hindi-tokenizer-output/splits/test.txt"       # Original text for BPB + generation
TOKENIZER_PATH = "/kaggle/input/datasets/srinivasu2005/hindi-tokenizer-output/tokenizer/tokenizer.model"
OUTPUT_DIR = "/kaggle/working/eval_hindi"

LANG = "Hindi"
VOCAB_SIZE = 8000
D_MODEL = 512; N_LAYERS = 8; N_HEADS = 8; D_FF = 1376
MAX_SEQ_LEN = 512; DROPOUT = 0.0   # No dropout at eval
TIE_WEIGHTS = True; ROPE_BASE = 10000
PAD_ID = 0; BOS_ID = 2; EOS_ID = 3

NUM_PREFIXES = 100; PREFIX_TOKENS = 32; MAX_NEW_TOKENS = 128
TEMPERATURES = [0.0, 0.5, 1.0, 1.5]; TOP_K = 50
BATCH_SIZE = 16; EVAL_MAX_BATCHES = None  # Full test set

In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "sacrebleu", "rouge-score", "matplotlib"])
try:
    subprocess.check_call(["wget", "-q", "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansDevanagari/NotoSansDevanagari-Regular.ttf", "-O", "/tmp/NotoSansDevanagari-Regular.ttf"])
except Exception as e:
    print(f"Warning: Could not fetch font. Error: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 9.3 MB/s eta 0:00:00


In [4]:
import os, math, json, time, logging
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from collections import Counter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
if os.path.exists('/tmp/NotoSansDevanagari-Regular.ttf'):
    fm.fontManager.addfont('/tmp/NotoSansDevanagari-Regular.ttf')
plt.rcParams['font.family'] = ['Noto Sans Devanagari', 'DejaVu Sans']
import sentencepiece as spm

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s — %(message)s")
logger = logging.getLogger("eval_hindi")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
@dataclass
class TransformerConfig:
    vocab_size: int = VOCAB_SIZE; d_model: int = D_MODEL; n_layers: int = N_LAYERS
    n_heads: int = N_HEADS; d_ff: int = D_FF; max_seq_len: int = MAX_SEQ_LEN
    dropout: float = DROPOUT; tie_weights: bool = TIE_WEIGHTS; rope_base: int = ROPE_BASE
    pad_id: int = PAD_ID; bos_id: int = BOS_ID; eos_id: int = EOS_ID
    @property
    def d_k(self): return self.d_model // self.n_heads

def precompute_rope_frequencies(d_k, max_seq_len, base=10000, device=None):
    freq = torch.arange(0, d_k, 2, dtype=torch.float32, device=device)
    theta = 1.0 / (base ** (freq / d_k))
    pos = torch.arange(0, max_seq_len, dtype=torch.float32, device=device)
    angles = torch.outer(pos, theta)
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    c = cos.unsqueeze(0).unsqueeze(0); s = sin.unsqueeze(0).unsqueeze(0)
    return torch.stack([x1*c - x2*s, x1*s + x2*c], dim=-1).flatten(-2)

class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads, self.d_k, self.d_model = cfg.n_heads, cfg.d_k, cfg.d_model
        self.W_Q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_K = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_V = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_O = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        cos, sin = precompute_rope_frequencies(cfg.d_k, cfg.max_seq_len, cfg.rope_base)
        self.register_buffer("cos_table", cos); self.register_buffer("sin_table", sin)
        self.register_buffer("causal_mask", torch.triu(torch.ones(cfg.max_seq_len, cfg.max_seq_len), diagonal=1).bool())
    def forward(self, x, return_attention=False):
        B, T, _ = x.shape
        Q = self.W_Q(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        K = self.W_K(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        V = self.W_V(x).view(B,T,self.n_heads,self.d_k).transpose(1,2)
        c, s = self.cos_table[:T], self.sin_table[:T]
        Q, K = apply_rope(Q,c,s), apply_rope(K,c,s)
        sc = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        sc = sc.masked_fill(self.causal_mask[:T,:T].unsqueeze(0).unsqueeze(0), float('-inf'))
        aw = torch.softmax(sc, dim=-1)
        ctx = torch.matmul(self.attn_dropout(aw), V).transpose(1,2).contiguous().view(B,T,self.d_model)
        return (self.resid_dropout(self.W_O(ctx)), aw) if return_attention else (self.resid_dropout(self.W_O(ctx)), None)

class SwiGLUFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.W_gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)
    def forward(self, x): return self.dropout(self.W_down(F.silu(self.W_gate(x)) * self.W_up(x)))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model); self.attn = MultiHeadCausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model); self.ffn = SwiGLUFeedForward(cfg)
    def forward(self, x, return_attention=False):
        ao, aw = self.attn(self.ln1(x), return_attention); x = x + ao; x = x + self.ffn(self.ln2(x)); return x, aw

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.emb_dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_ln = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.vocab_size if False else cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_weights: self.lm_head.weight = self.embedding.weight
    def forward(self, input_ids, targets=None, return_attention=False):
        x = self.emb_dropout(self.embedding(input_ids))
        all_attn = [] if return_attention else None
        for block in self.blocks:
            x, aw = block(x, return_attention)
            if return_attention and aw is not None: all_attn.append(aw)
        logits = self.lm_head(self.final_ln(x))
        result = {"logits": logits}
        if targets is not None:
            result["loss"] = F.cross_entropy(logits.view(-1, self.config.vocab_size), targets.view(-1), ignore_index=self.config.pad_id)
        if return_attention: result["attention_weights"] = all_attn
        return result
    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=128, temperature=1.0, top_k=0, repetition_penalty=1.0):
        self.eval(); gen = input_ids
        for _ in range(max_new_tokens):
            ctx = gen if gen.shape[1] <= self.config.max_seq_len else gen[:, -self.config.max_seq_len:]
            logits = self.forward(ctx)["logits"][:, -1, :]
            
            if repetition_penalty != 1.0:
                for b in range(input_ids.size(0)):
                    for token_id in set(gen[b].tolist()):
                        if logits[b, token_id] < 0:
                            logits[b, token_id] *= repetition_penalty
                        else:
                            logits[b, token_id] /= repetition_penalty
                            
            if temperature == 0.0: nt = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = float('-inf')
                nt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            gen = torch.cat([gen, nt], dim=1)
            if (nt == self.config.eos_id).all(): break
        return gen

class TokenizedDataset(Dataset):
    def __init__(self, bin_path, seq_len=512):
        self.seq_len = seq_len; self.data = np.memmap(bin_path, dtype=np.uint16, mode='r')
        self.n_samples = max(0, (len(self.data) - 1) // seq_len)
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        start = idx * self.seq_len
        c = self.data[start:start+self.seq_len+1].astype(np.int64)
        return torch.from_numpy(c[:-1]), torch.from_numpy(c[1:])
    def get_total_tokens(self): return len(self.data)

In [6]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
config = TransformerConfig(**ckpt["config"])
model = DecoderOnlyTransformer(config).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
logger.info(f"Loaded model from step {ckpt.get('step', '?')}, val_loss={ckpt.get('val_loss', '?')}")

sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
logger.info(f"Tokenizer: vocab={sp.get_piece_size()}")

all_results = {"language": LANG, "checkpoint_step": ckpt.get("step", -1)}

# ============================================================

[2026-09-02 12:50:17,139] INFO — Loaded model from step 16000, val_loss=3.3039690709114073
[2026-09-02 12:50:17,172] INFO — Tokenizer: vocab=8000


In [7]:
# ============================================================
logger.info("\n" + "="*60 + "\n1. Intrinsic LM Metrics\n" + "="*60)

test_ds = TokenizedDataset(TEST_BIN, MAX_SEQ_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, drop_last=True)

total_loss, total_tokens, n_batches = 0.0, 0, 0
with torch.no_grad():
    for x, y in test_loader:
        if EVAL_MAX_BATCHES and n_batches >= EVAL_MAX_BATCHES: break
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            result = model(x, targets=y)
        non_pad = (y != PAD_ID).sum().item()
        total_loss += result["loss"].item() * non_pad
        total_tokens += non_pad; n_batches += 1

avg_loss = total_loss / max(1, total_tokens)
ppl = math.exp(avg_loss) if avg_loss < 20 else float('inf')

# BPB
num_bytes = 0
if os.path.exists(TEST_TEXT):
    with open(TEST_TEXT, "rb") as f: num_bytes = len(f.read())
bpb = (avg_loss * test_ds.get_total_tokens() / num_bytes * math.log2(math.e)) if num_bytes > 0 else None

lm_results = {"test_loss": round(avg_loss, 4), "test_ppl": round(ppl, 2), "test_bpb": round(bpb, 4) if bpb else None,
              "num_tokens_evaluated": total_tokens, "num_bytes": num_bytes}
all_results["lm_metrics"] = lm_results
logger.info(f"  Loss: {avg_loss:.4f} | PPL: {ppl:.2f} | BPB: {bpb:.4f}" if bpb else f"  Loss: {avg_loss:.4f} | PPL: {ppl:.2f}")

# ============================================================

[2026-09-02 12:50:17,187] INFO — 
1. Intrinsic LM Metrics
[2026-09-02 12:53:20,492] INFO —   Loss: 3.2946 | PPL: 26.97 | BPB: 0.5101


In [8]:
# ============================================================
logger.info("\n" + "="*60 + "\n2. Generation Quality\n" + "="*60)

# Extract prefixes from test text
prefixes = []
if os.path.exists(TEST_TEXT):
    with open(TEST_TEXT, "r", encoding="utf-8") as f:
        doc = []
        for line in f:
            s = line.strip()
            if s: doc.append(s)
            elif doc:
                text = " ".join(doc)
                ids = sp.encode(text, out_type=int)
                if len(ids) >= PREFIX_TOKENS + MAX_NEW_TOKENS:
                    prefixes.append({"prefix_ids": ids[:PREFIX_TOKENS],
                                     "ref_text": sp.decode(ids[PREFIX_TOKENS:PREFIX_TOKENS+MAX_NEW_TOKENS])})
                    if len(prefixes) >= NUM_PREFIXES: break
                doc = []
    logger.info(f"  Extracted {len(prefixes)} prefixes")

gen_results = {}
for temp in TEMPERATURES:
    gen_ids_list = []
    gen_texts = []
    for p in prefixes:
        inp = torch.tensor([p["prefix_ids"]], dtype=torch.long, device=device)
        with torch.no_grad():
            out = model.generate(inp, max_new_tokens=MAX_NEW_TOKENS, temperature=temp, top_k=TOP_K if temp > 0 else 0, repetition_penalty=1.2)
        gids = out[0, len(p["prefix_ids"]):].tolist()
        gen_ids_list.append(gids)
        gen_texts.append(sp.decode(gids))
    
    ref_texts = [p["ref_text"] for p in prefixes]
    
    # Compute metrics
    metrics = {"num_samples": len(gen_texts)}
    try:
        import sacrebleu
        from rouge_score import rouge_scorer
        bleu = sacrebleu.corpus_bleu(gen_texts, [ref_texts])
        chrf = sacrebleu.corpus_chrf(gen_texts, [ref_texts], word_order=2)
        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
        rl = [scorer.score(r, g)['rougeL'].fmeasure for r, g in zip(ref_texts, gen_texts)]
        metrics.update({"bleu4": round(bleu.score, 2), "chrf_pp": round(chrf.score, 2),
                       "rouge_l": round(sum(rl)/len(rl)*100, 2) if rl else 0})
    except Exception as e:
        logger.warning(f"  Metrics error: {e}")
    
    metrics["sample_generations"] = gen_texts[:5]
    metrics["sample_references"] = ref_texts[:5]
    metrics["generated_ids"] = gen_ids_list  # for diversity analysis
    gen_results[f"temp_{temp}"] = metrics
    logger.info(f"  Temp={temp}: BLEU={metrics.get('bleu4','?')}, chrF++={metrics.get('chrf_pp','?')}, ROUGE-L={metrics.get('rouge_l','?')}")

all_results["generation"] = gen_results

# ============================================================

[2026-09-02 12:53:20,513] INFO — 
2. Generation Quality
[2026-09-02 12:53:20,607] INFO —   Extracted 100 prefixes
[2026-09-02 12:53:47,059] INFO — NumExpr defaulting to 4 threads.
[2026-09-02 12:53:48,019] INFO — Using default tokenizer.
[2026-09-02 12:53:48,022] INFO —   Temp=0.0: BLEU=0.02, chrF++=4.68, ROUGE-L=2.73
[2026-09-02 12:54:23,125] INFO — Using default tokenizer.
[2026-09-02 12:54:23,128] INFO —   Temp=0.5: BLEU=0.11, chrF++=6.31, ROUGE-L=1.4
[2026-09-02 12:55:37,232] INFO — Using default tokenizer.
[2026-09-02 12:55:37,235] INFO —   Temp=1.0: BLEU=0.39, chrF++=11.8, ROUGE-L=0.73
[2026-09-02 12:57:20,824] INFO — Using default tokenizer.
[2026-09-02 12:57:20,828] INFO —   Temp=1.5: BLEU=0.19, chrF++=14.16, ROUGE-L=0.33


In [9]:
# ============================================================
logger.info("\n" + "="*60 + "\n3. Diversity Diagnostics\n" + "="*60)

def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def repetition_rate(tokens, n=4):
    ng = get_ngrams(tokens, n)
    if not ng: return 0.0
    c = Counter(ng); return sum(v-1 for v in c.values() if v > 1) / len(ng)

def distinct_n(tokens, n):
    ng = get_ngrams(tokens, n)
    return len(set(ng)) / len(ng) if ng else 0.0

diversity_results = {}
for temp_key, gdata in gen_results.items():
    all_gen = gdata.get("generated_ids", [])
    rep = {f"rep_{n}gram": round(np.mean([repetition_rate(g,n) for g in all_gen if len(g)>n]),4) for n in [1,2,3,4]}
    dist = {f"distinct_{n}": round(np.mean([distinct_n(g,n) for g in all_gen if len(g)>n]),4) for n in [1,2]}
    diversity_results[temp_key] = {**rep, **dist}
    logger.info(f"  {temp_key}: D1={dist['distinct_1']:.3f}, D2={dist['distinct_2']:.3f}, Rep4={rep['rep_4gram']:.3f}")

all_results["diversity"] = diversity_results

# ============================================================

[2026-09-02 12:57:20,849] INFO — 
3. Diversity Diagnostics
[2026-09-02 12:57:20,854] INFO —   temp_0.0: D1=0.898, D2=0.931, Rep4=0.062
[2026-09-02 12:57:20,860] INFO —   temp_0.5: D1=0.890, D2=0.949, Rep4=0.033
[2026-09-02 12:57:20,870] INFO —   temp_1.0: D1=0.881, D2=0.987, Rep4=0.001
[2026-09-02 12:57:20,883] INFO —   temp_1.5: D1=0.889, D2=0.994, Rep4=0.000


In [10]:
# ============================================================
logger.info("\n" + "="*60 + "\n4. Attention Analysis\n" + "="*60)

attn_dir = os.path.join(OUTPUT_DIR, "attention")
os.makedirs(attn_dir, exist_ok=True)

# Pick a sample sentence
if prefixes:
    sample_ids = prefixes[0]["prefix_ids"][:40]
    sample_tokens = [sp.id_to_piece(i) for i in sample_ids]
    inp = torch.tensor([sample_ids], dtype=torch.long, device=device)
    
    with torch.no_grad():
        result = model(inp, return_attention=True)
    attn_weights_all = result["attention_weights"]
    
    # Heatmaps
    layers_to_plot = [0, N_LAYERS//2, N_LAYERS-1]
    for li in layers_to_plot:
        aw = attn_weights_all[li]  # (1, h, T, T)
        for hi in range(min(N_HEADS, 4)):
            a = aw[0, hi].cpu().numpy()
            fig, ax = plt.subplots(figsize=(10, 8))
            ax.imshow(a, cmap='Blues', aspect='auto', vmin=0)
            if len(sample_tokens) <= 40:
                ax.set_xticks(range(len(sample_tokens))); ax.set_yticks(range(len(sample_tokens)))
                ax.set_xticklabels(sample_tokens, rotation=90, fontsize=6)
                ax.set_yticklabels(sample_tokens, fontsize=6)
            ax.set_xlabel("Key"); ax.set_ylabel("Query")
            ax.set_title(f"{LANG} Layer {li}, Head {hi}")
            plt.colorbar(ax.images[0], ax=ax)
            plt.tight_layout()
            plt.savefig(os.path.join(attn_dir, f"heatmap_L{li}_H{hi}.png"), dpi=150)
            plt.close()
    
    # Entropy per head/layer
    entropy_data = {}
    for li, aw in enumerate(attn_weights_all):
        a = aw.clamp(min=1e-10)
        ent = -(a * a.log()).sum(dim=-1)  # (1, h, T)
        for hi in range(N_HEADS):
            entropy_data[f"L{li}_H{hi}"] = round(ent[0, hi].mean().item(), 4)
    
    # Mean attention distance
    distance_data = {}
    for li, aw in enumerate(attn_weights_all):
        T = aw.shape[-1]
        pos = torch.arange(T, device=device, dtype=torch.float32)
        dist_mat = (pos.unsqueeze(0) - pos.unsqueeze(1)).abs()
        md = (aw * dist_mat.unsqueeze(0).unsqueeze(0)).sum(dim=-1)  # (1, h, T)
        for hi in range(N_HEADS):
            distance_data[f"L{li}_H{hi}"] = round(md[0, hi].mean().item(), 4)
    
    # Summary heatmaps
    for data, name, cmap, label in [
        (entropy_data, "entropy_summary", "YlOrRd", "Entropy (nats)"),
        (distance_data, "distance_summary", "viridis", "Mean Distance (tokens)")
    ]:
        mat = np.zeros((N_LAYERS, N_HEADS))
        for k, v in data.items():
            l, h = int(k.split("_")[0][1:]), int(k.split("_")[1][1:])
            mat[l, h] = v
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(mat, cmap=cmap, aspect='auto')
        ax.set_xlabel("Head"); ax.set_ylabel("Layer")
        ax.set_title(f"{LANG} {label}")
        ax.set_xticks(range(N_HEADS)); ax.set_yticks(range(N_LAYERS))
        plt.colorbar(im, ax=ax, label=label)
        plt.tight_layout()
        plt.savefig(os.path.join(attn_dir, f"{name}.png"), dpi=150)
        plt.close()
    
    all_results["attention"] = {"entropy": entropy_data, "mean_distance": distance_data}
    logger.info(f"  Saved {len(os.listdir(attn_dir))} attention plots to {attn_dir}/")

# ============================================================

[2026-09-02 12:57:20,903] INFO — 
4. Attention Analysis
[2026-09-02 12:57:28,374] INFO —   Saved 14 attention plots to /kaggle/working/eval_hindi/attention/


In [11]:
# ============================================================
# Clean up non-serializable data
for tk in list(gen_results.keys()):
    if "generated_ids" in gen_results[tk]:
        del gen_results[tk]["generated_ids"]

results_path = os.path.join(OUTPUT_DIR, "eval_results.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
logger.info(f"\nAll results saved to {results_path}")

[2026-09-02 12:57:28,391] INFO — 
All results saved to /kaggle/working/eval_hindi/eval_results.json


In [12]:
print("\n" + "="*60)
print(f"  {LANG} Model Evaluation Summary")
print("="*60)
print(f"  Test Loss:  {lm_results['test_loss']}")
print(f"  Test PPL:   {lm_results['test_ppl']}")
print(f"  Test BPB:   {lm_results.get('test_bpb', 'N/A')}")
print("-"*60)
for tk, m in gen_results.items():
    print(f"  {tk}: BLEU={m.get('bleu4','?')}, chrF++={m.get('chrf_pp','?')}, ROUGE-L={m.get('rouge_l','?')}")
print("-"*60)
for tk, d in diversity_results.items():
    print(f"  {tk}: D1={d['distinct_1']:.3f}, D2={d['distinct_2']:.3f}")
print("="*60)


  Hindi Model Evaluation Summary
  Test Loss:  3.2946
  Test PPL:   26.97
  Test BPB:   0.5101
------------------------------------------------------------
  temp_0.0: BLEU=0.02, chrF++=4.68, ROUGE-L=2.73
  temp_0.5: BLEU=0.11, chrF++=6.31, ROUGE-L=1.4
  temp_1.0: BLEU=0.39, chrF++=11.8, ROUGE-L=0.73
  temp_1.5: BLEU=0.19, chrF++=14.16, ROUGE-L=0.33
------------------------------------------------------------
  temp_0.0: D1=0.898, D2=0.931
  temp_0.5: D1=0.890, D2=0.949
  temp_1.0: D1=0.881, D2=0.987
  temp_1.5: D1=0.889, D2=0.994
